# Interactive Storm Tracks Visualization

This notebook provides an interactive Cartopy map to visualize storm tracks from the PyStormTracker output.

In [ ]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import pandas as pd
import os
from pystormtracker.io.imilast import read_imilast
from pystormtracker.utils.geo import geod_dist_km

In [ ]:
# Load track data - Defaulting to MSL file
track_file = "../data/test/tracks/era5_vo_2.5x2.5_1e-4_v0.0.2_imilast.txt"
tracks = read_imilast(track_file)

# Detect track type (VO or MSL)
track_type = 'vo' if 'vo' in os.path.basename(track_file).lower() else 'msl'
print(f"Detected track type: {track_type.upper()}")

In [ ]:
# Convert to Pandas DataFrame
raw_intensity = tracks.vars.get('Intensity1', np.zeros_like(tracks.lats))

data = {
    'track_id': tracks.track_ids,
    'time': tracks.times,
    'lat': tracks.lats,
    'lon': tracks.lons,
    'intensity': raw_intensity
}
df = pd.DataFrame(data)

# Apply scaling factors for user-friendly units
if track_type == 'vo':
    # Scale VO to 10^-5 units
    df['display_intensity'] = df['intensity'] * 1e5
    intensity_unit = "10⁻⁵ s⁻¹"
else:
    # MSL: Convert Pa to hPa
    df['display_intensity'] = df['intensity'] / 100.0
    intensity_unit = "hPa"

# Sort for correct distance/duration calculation
df = df.sort_values(['track_id', 'time'])

In [ ]:
# Pre-calculate track-level statistics
print("Calculating track statistics...")
grouped = df.groupby('track_id')

if track_type == 'vo':
    track_strength = grouped['display_intensity'].max().to_dict()
    strength_desc = f"Min Peak VO ({intensity_unit})"
else:
    track_strength = grouped['display_intensity'].min().to_dict()
    strength_desc = f"Max Peak MSL ({intensity_unit})"
df['track_strength'] = df['track_id'].map(track_strength)

track_durations = (grouped['time'].max() - grouped['time'].min()) / np.timedelta64(1, 'h')
df['track_duration_hrs'] = df['track_id'].map(track_durations.to_dict())

def calc_displacement(g):
    return geod_dist_km(g.iloc[0]['lat'], g.iloc[0]['lon'], g.iloc[-1]['lat'], g.iloc[-1]['lon'])
track_displacements = grouped.apply(calc_displacement)
df['track_displacement_km'] = df['track_id'].map(track_displacements.to_dict())

unique_times = np.sort(df['time'].unique())
print(f"Loaded {len(tracks.track_ids)} points, {len(grouped)} tracks.")

In [ ]:
# Setup User Interface (Plot + Sliders)

# 1. Create the Figure
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
ax.set_global()

lc = LineCollection([], colors='red', linewidths=1.0, alpha=0.7, transform=ccrs.PlateCarree())
ax.add_collection(lc)
title_text = ax.set_title("Storm Tracks")

# 2. Create the Widgets
time_range_slider = widgets.SelectionRangeSlider(
    options=list(unique_times),
    index=(0, len(unique_times) - 1),
    description='Time Range:',
    layout={'width': '600px'}
)

min_di, max_di = df['display_intensity'].min(), df['display_intensity'].max()
strength_slider = widgets.IntSlider(
    value=int(max_di) if track_type == 'msl' else int(min_di),
    min=int(min_di), max=int(max_di), step=1,
    description=strength_desc, layout={'width': '600px'}
)

duration_slider = widgets.IntSlider(
    value=42, min=0, max=int(df['track_duration_hrs'].max()),
    step=6, description='Min Dur (h):', layout={'width': '600px'}
)

dist_slider = widgets.IntSlider(
    value=1000, min=0, max=int(df['track_displacement_km'].max()),
    step=100, description='Min Dist (km):', layout={'width': '600px'}
)

# 3. Update Logic
def update_plot(change):
    t_start, t_end = time_range_slider.value
    threshold = strength_slider.value
    min_dur = duration_slider.value
    min_dist = dist_slider.value
    
    if track_type == 'vo':
        mask = (df['track_strength'] >= threshold)
    else:
        mask = (df['track_strength'] <= threshold)
    
    mask &= (df['track_duration_hrs'] >= min_dur)
    mask &= (df['track_displacement_km'] >= min_dist)
    
    eligible_tracks = df[mask]['track_id'].unique()
    
    filtered_df = df[
        (df['track_id'].isin(eligible_tracks)) & 
        (df['time'] >= t_start) & 
        (df['time'] <= t_end)
    ]
    
    lines = []
    if not filtered_df.empty:
        for tid, group in filtered_df.groupby('track_id'):
            if len(group) > 1:
                lons, lats = group['lon'].values, group['lat'].values
                diffs = np.diff(lons)
                splits = np.where(np.abs(diffs) > 180)[0] + 1
                segments_lon = np.split(lons, splits)
                segments_lat = np.split(lats, splits)
                for s_lon, s_lat in zip(segments_lon, segments_lat):
                    if len(s_lon) > 1:
                        lines.append(np.c_[s_lon, s_lat])
    
    lc.set_segments(lines)
    
    s_str = pd.to_datetime(t_start).strftime('%Y-%m-%d %H:%M')
    e_str = pd.to_datetime(t_end).strftime('%Y-%m-%d %H:%M')
    cond = ">=" if track_type == 'vo' else "<="
    
    title_text.set_text(
        f"{track_type.upper()} Tracks: {s_str} to {e_str}\n"
        f"Peak {cond} {threshold} {intensity_unit} | Dur >= {min_dur}h | Dist >= {min_dist}km"
    )
    fig.canvas.draw_idle()

# 4. Display UI and Initial Update
for w in [time_range_slider, strength_slider, duration_slider, dist_slider]:
    w.observe(update_plot, names='value')

ui = widgets.VBox([time_range_slider, strength_slider, duration_slider, dist_slider])
display(ui)
update_plot(None)